In [9]:
# import packages and load data in
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

year_and_month = ['202511','202512','202601', '202602', '202603', '202604', '202605']
CRMLSSold_data = pd.DataFrame()

for year_month in year_and_month:
    # Load the data for each year and month
    data = pd.read_csv(f'data/CRMLSSold{year_month}.csv')
    # Append the data to the main DataFrame
    CRMLSSold_data = pd.concat([CRMLSSold_data, data], ignore_index=True)

# --- PropertyType filter ---
CRMLSSold_data = CRMLSSold_data[
    (CRMLSSold_data['PropertyType'] == 'Residential') &
    (CRMLSSold_data['PropertySubType'] == 'SingleFamilyResidence')
]

/var/folders/jc/zlyx5cfj47b4kf9kgt678kkc0000gn/T/ipykernel_79553/2024757023.py:12: DtypeWarning: Columns (0: WaterfrontYN, 1: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(f'data/CRMLSSold{year_month}.csv')


In [10]:
# Features decided in 01_exploration.ipynb

# Target
target = ['ClosePrice']  # log-transform for modeling

# Core size/structure features
size_features = [
    'LivingArea',           # log-transform; strongest predictor
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'YearBuilt',            # or bin into decades / home age
    'GarageSpaces',         # ~13% missing; impute
    'LotSizeSquareFeet'    # weak standalone signal; include anyway

]

# Location features
location_features = [
    'Latitude',
    'Longitude',
    'CountyOrParish',       # alternative categorical encoding
]

# Amenity flags
amenity_features = [
    'PoolPrivateYN',
    'ViewYN',
    'FireplaceYN',
    # 'NewConstructionYN',  # direction counterintuitive; investigate before using
    # 'WaterfrontYN',       # excluded: only 86 non-missing rows
]

features = size_features + location_features + amenity_features
print(f"Total features: {len(features)}")
print(features)

Total features: 12
['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'YearBuilt', 'GarageSpaces', 'LotSizeSquareFeet', 'Latitude', 'Longitude', 'CountyOrParish', 'PoolPrivateYN', 'ViewYN', 'FireplaceYN']


In [11]:
all_cols = target + features
missing = CRMLSSold_data[all_cols].isna().mean().mul(100).round(2).sort_values(ascending=False)
display(missing.rename('% Missing').to_frame())

,% Missing
ViewYN,8.69
PoolPrivateYN,7.81
GarageSpaces,3.85
LotSizeSquareFeet,1.72
FireplaceYN,0.09
YearBuilt,0.06
LivingArea,0.05
Latitude,0.01
Longitude,0.01
ClosePrice,0.00


In [12]:
# Sanity filters and 1st/99th percentile boundary removal

n_start = len(CRMLSSold_data)
print(f"Start: {n_start:,}")

# --- Hard filters: physically impossible or clearly bad values ---
n = len(CRMLSSold_data)
CRMLSSold_data = CRMLSSold_data[
    (CRMLSSold_data['ClosePrice'] > 0) &
    (CRMLSSold_data['LivingArea'] > 0) &
    (CRMLSSold_data['BedroomsTotal'] > 0) &
    (CRMLSSold_data['YearBuilt'].isna() | CRMLSSold_data['YearBuilt'].between(1800, 2026))
]
print(f"After hard filters:      {len(CRMLSSold_data):,}  (-{n - len(CRMLSSold_data):,})")

# --- 1st/99th percentile removal for heavily skewed continuous features ---
# Compute all bounds first on the same snapshot, then filter once.
price_low,  price_high  = CRMLSSold_data['ClosePrice'].quantile([0.01, 0.99])
living_low, living_high = CRMLSSold_data['LivingArea'].quantile([0.01, 0.99])
lot_low,    lot_high    = CRMLSSold_data['LotSizeSquareFeet'].quantile([0.01, 0.99])

n = len(CRMLSSold_data)
CRMLSSold_data = CRMLSSold_data[
    CRMLSSold_data['ClosePrice'].between(price_low, price_high) &
    CRMLSSold_data['LivingArea'].between(living_low, living_high) &
    (CRMLSSold_data['LotSizeSquareFeet'].isna() | CRMLSSold_data['LotSizeSquareFeet'].between(lot_low, lot_high))
]
print(f"After percentile filter: {len(CRMLSSold_data):,}  (-{n - len(CRMLSSold_data):,})")

n_end = len(CRMLSSold_data)
print(f"\n{n_start:,}  →  {n_end:,}  (-{n_start - n_end:,}, {(n_start - n_end) / n_start:.1%})")
print(f"ClosePrice:        [{price_low:,.0f}, {price_high:,.0f}]")
print(f"LivingArea:        [{living_low:,.0f}, {living_high:,.0f}]")
print(f"LotSizeSquareFeet: [{lot_low:,.0f}, {lot_high:,.0f}]")
CRMLSSold_data = CRMLSSold_data.reset_index(drop=True)

Start: 71,466
After hard filters:      71,383  (-83)
After percentile filter: 67,706  (-3,677)

71,466  →  67,706  (-3,760, 5.3%)
ClosePrice:        [230,000, 6,450,000]
LivingArea:        [743, 5,686]
LotSizeSquareFeet: [1,724, 306,227]


In [13]:
all_cols = target + features
missing = CRMLSSold_data[all_cols].isna().mean().mul(100).round(2).sort_values(ascending=False)
display(missing.rename('% Missing').to_frame())

,% Missing
ViewYN,8.98
PoolPrivateYN,7.63
GarageSpaces,3.65
LotSizeSquareFeet,1.76
FireplaceYN,0.08
YearBuilt,0.04
Latitude,0.01
Longitude,0.01
ClosePrice,0.00
LivingArea,0.00


In [14]:
# Impute amenity flags with 'Unknown' if missing as it could be a signal(only 10% of data)
# The missing values in these columns differentiate in median compared to 
# the non-missing values so this can indicate a signal or pattern that could be useful for modeling.
for col in ['ViewYN', 'PoolPrivateYN', 'FireplaceYN']:
    CRMLSSold_data[col] = CRMLSSold_data[col].fillna('Unknown')



# Drop rows where LotSizeSquareFeet is null and GarageSpaces is null (~5%)
# less than 5% so we are not losing a lot of data from removing these missing rows
n = len(CRMLSSold_data)
CRMLSSold_data = CRMLSSold_data.dropna(subset=all_cols)
print(f"After dropping null LotSizeSquareFeet: {len(CRMLSSold_data):,}  (-{n - len(CRMLSSold_data):,})")
# % difference 
print(f"Percentage of rows dropped: {(n - len(CRMLSSold_data)) / n:.2%}")

# Verify no remaining nulls in used columns
display(CRMLSSold_data[all_cols].isna().mean().mul(100).round(2).sort_values(ascending=False).rename('% Missing').to_frame())

# 

After dropping null LotSizeSquareFeet: 64,021  (-3,685)
Percentage of rows dropped: 5.44%


,% Missing
ClosePrice,0.0
LivingArea,0.0
BedroomsTotal,0.0
BathroomsTotalInteger,0.0
YearBuilt,0.0
GarageSpaces,0.0
LotSizeSquareFeet,0.0
Latitude,0.0
Longitude,0.0
CountyOrParish,0.0


In [15]:
from sklearn.preprocessing import StandardScaler
"""
Log transforms 

We log-transform to aid normality and skewness for our base linear regression model
non-linear variables still will work even with log close price as
it just changes scale of target variable
close price has strong correlation with living area so log-transform to make the relationship more linear
"""
CRMLSSold_data['LogClosePrice'] = np.log(CRMLSSold_data['ClosePrice'])
CRMLSSold_data['LogLivingArea'] = np.log(CRMLSSold_data['LivingArea'])

#  Categorical encoding 
cat_cols = ['CountyOrParish', 'ViewYN', 'PoolPrivateYN', 'FireplaceYN']
CRMLSSold_data = pd.get_dummies(CRMLSSold_data, columns=cat_cols, dtype=int)

#  Feature matrix 
log_features = ['LogLivingArea', 'BedroomsTotal', 'BathroomsTotalInteger',
                'YearBuilt', 'GarageSpaces', 'LotSizeSquareFeet',
                'Latitude', 'Longitude']
encoded_cols = [c for c in CRMLSSold_data.columns
                if c.startswith(('CountyOrParish_', 'ViewYN_', 'PoolPrivateYN_', 'FireplaceYN_'))]
feature_cols = log_features + encoded_cols

#  Train/test split by time 
# Test set: May 2026 (fixed). Train set: all preceding months in the dataset.
# about a 80/20 split of our data 
CRMLSSold_data['CloseDate'] = pd.to_datetime(CRMLSSold_data['CloseDate'])
CRMLSSold_data['CloseMonth'] = CRMLSSold_data['CloseDate'].dt.to_period('M')

TEST_MONTH = pd.Period('2026-05', freq='M')
train = CRMLSSold_data[CRMLSSold_data['CloseMonth'] < TEST_MONTH].copy()
test  = CRMLSSold_data[CRMLSSold_data['CloseMonth'] == TEST_MONTH].copy()

# Normalization to compare features on the same scale
# Fit scaler on train only, transform both — prevents leakage from test set
scale_cols = ['LogLivingArea', 'BedroomsTotal', 'BathroomsTotalInteger',
              'YearBuilt', 'GarageSpaces', 'LotSizeSquareFeet', 'Latitude', 'Longitude']
scaler = StandardScaler()
train[scale_cols] = scaler.fit_transform(train[scale_cols])
test[scale_cols]  = scaler.transform(test[scale_cols])

X_train = train[feature_cols]
y_train = train['LogClosePrice']
X_test  = test[feature_cols]
y_test  = test['LogClosePrice']


print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"Features: {len(feature_cols)}")

X_train: (53237, 76)  |  X_test: (10784, 76)
Features: 76


In [16]:
# sample of the training data
display(X_train.head())

,LogLivingArea,BedroomsTotal,BathroomsTotalInteger,YearBuilt,GarageSpaces,LotSizeSquareFeet,Latitude,Longitude,CountyOrParish_Alameda,CountyOrParish_Amador,...,CountyOrParish_Yuba,ViewYN_False,ViewYN_True,ViewYN_Unknown,PoolPrivateYN_False,PoolPrivateYN_True,PoolPrivateYN_Unknown,FireplaceYN_False,FireplaceYN_True,FireplaceYN_Unknown
0,-1.519033,-0.553081,-0.600547,-0.590186,-0.000145,-0.292734,-0.598746,0.157287,0,0,...,0,1,0,0,1,0,0,1,0,0
1,-1.275036,-0.553081,-1.625368,-1.032099,-0.000145,0.179615,1.402073,-0.908114,0,0,...,0,1,0,0,0,0,1,0,1,0
2,-0.346166,0.551856,0.424273,0.035857,-0.000145,-0.315863,-1.218895,0.409056,0,0,...,0,1,0,0,1,0,0,0,1,0
3,1.130847,1.656793,0.424273,0.846031,0.428590,-0.316919,-1.212878,0.419864,0,0,...,0,1,0,0,1,0,0,0,1,0
4,-0.179764,-0.553081,-0.600547,-0.995273,0.428590,-0.119664,0.448936,-0.576609,0,0,...,0,1,0,0,1,0,0,0,1,0


In [17]:
# save clean csv
CRMLSSold_data.to_csv('cleaned_CRMLSSOLD_data.csv', index=False)